In [ ]:
# 263A MSOL Project
#
# First design your robot in Fusion 360 and export it into urdf file using the following link:
# https://www.youtube.com/watch?v=_ZFo6wPXjeQ
#(use fusion2urdf-ros2)
#
# 1) Setup & Imports
# 2) XML Templates / Robot Config (.xacro)
# 3) Utilities & Controller
# 4) Build/Run Functions
# 5) Execute

# 1. Setup & Dependencies (Google Colab)

Install required packages for Google Colab and mount Google Drive to access robot URDF files.

In [ ]:
# 1a) Install packages (Google Colab)
# Run this cell FIRST, then run the next cell for imports

# Install EGL for headless rendering (required for MuJoCo on Colab)
!apt-get install -y libegl1-mesa-dev libgl1-mesa-glx libgles2-mesa-dev

# Install Python packages
!pip install mujoco mediapy xacro urdf2mjcf pillow

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1b) Import libraries
# Run this cell AFTER running the installation cell above

import os

# Set headless rendering for Google Colab (MUST be before importing mujoco)
os.environ["MUJOCO_GL"] = "egl"

import time
import math
import re
import tempfile
import numpy as np
import xml.etree.ElementTree as ET
from scipy.spatial.transform import Rotation as R

import mujoco
import mediapy as media
HAVE_MUJOCO = True
print("// mujoco and mediapy loaded successfully (EGL headless mode)!")

import xacro
HAVE_XACRO = True
print("// xacro loaded successfully!")

from urdf2mjcf import run as urdf2mjcf_run
HAVE_URDF2MJCF = True
print("// urdf2mjcf loaded successfully!")

# 2. Environment & Robot Configuration

Defines the MuJoCo world (Death Star Hangar): floor, ceiling, walls, target box, hanging obstacle, and power cell. Placeholders inject custom robot body, assets, defaults, and actuators from URDF conversion. 

**Important**: Update `ROBOT_XACRO_PATH` and `ROBOT_PACKAGE_DIRS` to point to your robot files in Google Drive.

In [ ]:
# 2) Environment / Robot XML Templates
#
# ENV_XML_TEMPLATE: MuJoCo Death Star Hangar scene. Placeholders (ROBOT_*, INSIDE_SPHERE_OBJ_PATH,
# CONCRETE_TEX_PATH) are replaced at build time.
#   <option>: timestep (0.002s), gravity (0 0 -9.81)
#   <visual>: headlight, haze, render resolution
#   <asset>: textures (sky, stars, concrete), materials, meshes
#   <worldbody>: floor, ceiling, walls, space sphere, cameras, red_wall, target_box, power_cell, robot_base
#   ROBOT_ACTUATOR_PLACEHOLDER: joint actuators from URDF
#   <equality>: magnet weld (ee_site <-> power_cell_site) when grasp active

ENV_XML_TEMPLATE = """
<mujoco model="DeathStar_Hangar">
    <!-- ROBOT_COMPILER_PLACEHOLDER -->
    <option timestep="0.002" gravity="0 0 -9.81" integrator="implicitfast"/>

    <visual>
        <headlight diffuse="1.2 1.2 1.3" ambient="0.6 0.6 0.7" specular="0.3 0.3 0.4"/>
        <rgba haze="0.02 0.02 0.05 1"/>
        <global azimuth="100" elevation="-15" offwidth="960" offheight="720"/>
    </visual>

    <default>
        <joint damping="5" armature="0.1"/>
        <geom friction="1 0.5 0.5"/>
    </default>
    <!-- ROBOT_DEFAULT_PLACEHOLDER -->
    <asset>
        <!-- ROBOT_ASSET_PLACEHOLDER -->
        <texture name="space_skybox" type="skybox" builtin="gradient" rgb1="0.02 0.02 0.05" rgb2="0 0 0" width="256" height="256"/>
        <texture name="space_tex" type="2d" file="8k_stars_milky_way.png" width="1024" height="512"/>
        <material name="space_mat" texture="space_tex" texrepeat="1 1" reflectance="0.05"/>
        <mesh name="inside_sphere_mesh" file="INSIDE_SPHERE_OBJ_PATH" scale="50 50 50"/>
        <texture name="concrete_tex" type="2d" file="CONCRETE_TEX_PATH" width="1024" height="1024"/>
        <material name="concrete_mat" texture="concrete_tex" texrepeat="2 2" texuniform="true" reflectance="0.01"/>
        <material name="floor_mat" rgba="0.03 0.03 0.08 1" reflectance="0.02"/>
        <material name="metal" rgba="0.8 0.8 0.8 1" shininess="0.8" reflectance="0.5"/>
        <material name="obstacle" rgba="0.7 0.2 0.2 1"/>
    </asset>

    <worldbody>
        <geom name="floor" size="2 2 .1" type="plane" material="concrete_mat"/>
        <geom name="ceiling" size="2 2 .1" type="plane" pos="0 0 2.0" zaxis="0 0 -1" material="concrete_mat"/>

        <body name="wall_pos_y" pos="0 2 1.0">
            <geom type="box" size="0.1725 0.08625 1.035" pos="1.8275 0 0" material="metal"/>
            <geom type="box" size="0.1725 0.08625 1.035" pos="-1.8275 0 0" material="metal"/>
            <geom type="box" size="2.0 0.08625 0.1725" pos="0 0 0" material="metal"/>
        </body>
        <body name="wall_neg_y" pos="0 -2 1.0">
            <geom type="box" size="0.1725 0.08625 1.035" pos="1.8275 0 0" material="metal"/>
            <geom type="box" size="0.1725 0.08625 1.035" pos="-1.8275 0 0" material="metal"/>
            <geom type="box" size="2.0 0.08625 0.1725" pos="0 0 0" material="metal"/>
        </body>
        <geom name="space_sphere" type="mesh" mesh="inside_sphere_mesh" pos="0 0 1.0" material="space_mat" contype="0" conaffinity="0"/>

        <body name="cam_target" pos="0 0 0.5">
            <site name="cam_target_site" size="0.001" rgba="1 0 0 0"/>
        </body>

        <light pos="0 0 4" dir="0 0 -1" directional="true" diffuse="1 1 1" specular="0.5 0.5 0.5"/>
        <light pos="0.5 -0.5 2" dir="0 0 -1" directional="true" diffuse="0.6 0.6 0.7"/>

        <camera name="diag_cam" pos="0 -3.5 1.0" mode="targetbody" target="cam_target"/>
        <camera name="power_cam" pos="0 -2.0 0.8" mode="targetbody" target="power_cell"/>

        <body name="red_wall" pos="0.5 0 1.2">
            <geom name="red_wall_geom" type="box" size="0.05 0.8 0.6" rgba="0.9 0.1 0.1 1"/>
        </body>

        <body name="target_box" pos="1.2 0 0.15">
            <geom name="target_box_geom" type="box" size="0.12 0.12 0.15" rgba="0.3 0.6 0.3 1"/>
            <site name="target_center" pos="0 0 0.16" rgba="0 1 0 0.5" size="0.03"/>
        </body>

        <body name="power_cell" pos="0.4 -0.1 0.25">
            <freejoint/>
            <geom name="power_cell_main" type="cylinder" size="0.04 0.04" rgba="0.4 0.5 1 1" mass="0.2"/>
            <geom type="cylinder" size="0.02 0.04" pos="0 0 0.08" rgba="0.4 0.5 1 1" mass="0.1"/>
            <site name="power_cell_site" pos="0 0 0.04" size="0.01" rgba="0 1 0 0.5"/>
        </body>

        <body name="robot_base" pos="0.75 0.25 0.0">
            <site name="attachment_site" pos="0 0 0" size="0.01" rgba="1 0 0 1"/>
            <!-- ROBOT_PLACEHOLDER -->
        </body>
    </worldbody>

    <!-- ROBOT_ACTUATOR_PLACEHOLDER -->

    <equality>
        <weld name="magnet" site1="ee_site" site2="power_cell_site" active="false"/>
    </equality>
</mujoco>
"""

# ========================================
# TODO: Update these paths for your Google Drive
# ========================================
# Example: If your robot files are in "My Drive/C263A/robot urdf3/robot_description/"
# then set GDRIVE_PROJECT_PATH = "/content/drive/MyDrive/C263A"

GDRIVE_PROJECT_PATH = "/content/drive/MyDrive/C263A/Project 26W-1"  # <-- Change this to your Google Drive path

# Path to the robot .xacro file (relative to GDRIVE_PROJECT_PATH)
ROBOT_XACRO_PATH = os.path.join(GDRIVE_PROJECT_PATH, "robot urdf3/robot_description/urdf/robot.xacro")

# Maps package names to absolute paths for $(find pkg) and package:// resolution (ROS-free)
ROBOT_PACKAGE_DIRS = {"robot_description": os.path.join(GDRIVE_PROJECT_PATH, "robot urdf3/robot_description")}

# Set working directory to project path for asset files
os.chdir(GDRIVE_PROJECT_PATH)


def ensure_inside_sphere_obj(path, stacks=32, slices=64):
    if os.path.exists(path):
        return

    lines = []
    for i in range(stacks + 1):
        v = i / stacks
        theta = v * math.pi
        sin_t = math.sin(theta)
        cos_t = math.cos(theta)
        for j in range(slices + 1):
            u = j / slices
            phi = u * 2.0 * math.pi
            sin_p = math.sin(phi)
            cos_p = math.cos(phi)
            x = sin_t * cos_p
            y = sin_t * sin_p
            z = cos_t
            lines.append(f"v {x:.6f} {y:.6f} {z:.6f}\n")
            lines.append(f"vt {u:.6f} {1.0 - v:.6f}\n")

    def vid(i, j):
        return i * (slices + 1) + j + 1

    for i in range(stacks):
        for j in range(slices):
            v1 = vid(i, j)
            v2 = vid(i + 1, j)
            v3 = vid(i + 1, j + 1)
            v4 = vid(i, j + 1)
            lines.append(f"f {v1}/{v1} {v4}/{v4} {v3}/{v3}\n")
            lines.append(f"f {v1}/{v1} {v3}/{v3} {v2}/{v2}\n")

    with open(path, "w") as f:
        f.writelines(lines)

# 3. Utilities & Pick-and-Place Controller

`load_robot_from_xacro()` converts URDF to MJCF and extracts worldbody, defaults, assets, actuators, and joint names. `ensure_inside_sphere_obj()` generates the skybox mesh. `robot_controller()` implements IK-based pick-and-place: approach power cell → grasp (activate weld when ee within 2 cm) → carry to target box → release (deactivate weld when within 5 cm).

In [ ]:
# 3) Utilities & Robot Controller

def _resolve_package_urdf(urdf_str, package_dirs):
    """Replace package://pkg_name/... and $(find pkg)/... with actual paths."""
    if not package_dirs:
        return urdf_str
    def repl_pkg(m):
        pkg = m.group(1)
        rest = m.group(2)
        if pkg in package_dirs:
            base = package_dirs[pkg].rstrip("/")
            return base + "/" + rest.lstrip("/")
        return m.group(0)
    urdf_str = re.sub(r'package://([^/]+)/(.*?)(?="|\\s|$)', repl_pkg, urdf_str)
    # $(find pkg)/path
    def repl_find(m):
        pkg = m.group(1)
        rest = m.group(2)
        if pkg in package_dirs:
            base = package_dirs[pkg].rstrip("/")
            return base + "/" + rest.lstrip("/")
        return m.group(0)
    urdf_str = re.sub(r'\$\(find\s+([^\s)]+)\)/(.*?)(?="|\s|\))', repl_find, urdf_str)
    # urdf2mjcf may treat file:// paths as relative strings; normalize to local absolute paths.
    urdf_str = urdf_str.replace("file:///", "/")
    urdf_str = urdf_str.replace("file://", "")
    urdf_str = urdf_str.replace("file:/", "/")
    return urdf_str


def _resolve_find_in_xacro(content, package_dirs):
    """Pre-process xacro content: replace $(find pkg) for ROS-free use."""
    if not package_dirs:
        return content
    def repl(m):
        pkg = m.group(1)
        if pkg in package_dirs:
            return package_dirs[pkg].rstrip("/")
        return m.group(0)
    return re.sub(r'\$\(find\s+([^\s)]+)\)', repl, content)


def load_robot_from_xacro(xacro_path, package_dirs=None):
    """
    Load robot from .xacro file.
    Returns MJCF XML string (body content to inject into robot_base).
    Requires: pip install xacro urdf2mjcf
    """
    if not HAVE_XACRO:
        raise RuntimeError("xacro not installed. Run: pip install xacro")
    if not HAVE_URDF2MJCF:
        raise RuntimeError("urdf2mjcf not installed. Run: pip install urdf2mjcf")

    abs_path = os.path.abspath(xacro_path)
    if not os.path.exists(abs_path):
        raise FileNotFoundError(f"xacro file not found: {abs_path}")

    work_dir = os.path.dirname(abs_path)

    # 0) Pre-process: $(find pkg) -> actual path (xacro uses this in includes)
    with open(abs_path, "r") as f:
        xacro_content = _resolve_find_in_xacro(f.read(), package_dirs)
    xacro_preprocessed = os.path.join(work_dir, "_temp_robot_preprocessed.xacro")
    urdf_file = os.path.join(work_dir, "_temp_robot.urdf")
    mjcf_file = os.path.join(work_dir, "_temp_robot.mjcf")
    with open(xacro_preprocessed, "w") as f:
        f.write(xacro_content)
    try:
        # 1) xacro -> URDF
        doc = xacro.process_file(xacro_preprocessed)
        urdf_str = doc.toxml()
        urdf_str = _resolve_package_urdf(urdf_str, package_dirs)

        with open(urdf_file, "w") as f:
            f.write(urdf_str)

        # 2) URDF -> MJCF
        try:
            urdf2mjcf_run(
                urdf_path=urdf_file,
                mjcf_path=mjcf_file,
                copy_meshes=True,
            )
        except TypeError as e:
            # Compatibility path: older SciPy does not support Rotation.as_quat(scalar_first=...)
            if "scalar_first" not in str(e):
                raise
            from scipy.spatial.transform import Rotation as _SciPyRotation
            import urdf2mjcf.postprocess.add_sensors as _add_sensors

            class _RotationCompat:
                @staticmethod
                def from_euler(*args, **kwargs):
                    rot = _SciPyRotation.from_euler(*args, **kwargs)

                    class _WrappedRotation:
                        def __init__(self, r):
                            self._r = r

                        def as_quat(self, scalar_first=False):
                            q = self._r.as_quat()
                            if scalar_first:
                                return np.array([q[3], q[0], q[1], q[2]])
                            return q

                    return _WrappedRotation(rot)

            _add_sensors.R = _RotationCompat
            urdf2mjcf_run(
                urdf_path=urdf_file,
                mjcf_path=mjcf_file,
                copy_meshes=True,
            )

        with open(mjcf_file, "r") as f:
            mjcf_str = f.read()

        # 3) Extract robot-related MJCF blocks from conversion output (XML-safe)
        try:
            root = ET.fromstring(mjcf_str)
        except ET.ParseError as pe:
            raise RuntimeError(f"Failed to parse urdf2mjcf output as XML: {pe}")

        def _to_xml(elem):
            return ET.tostring(elem, encoding="unicode")

        # worldbody: inject only inner children
        worldbody_node = root.find("worldbody")
        if worldbody_node is None:
            raise RuntimeError("Could not extract worldbody from urdf2mjcf output")
        worldbody_inner = "\n".join(_to_xml(child) for child in list(worldbody_node)).strip()

        # defaults: keep top-level default blocks exactly, preserving nested defaults
        default_nodes = root.findall("default")
        default_xml = "\n".join(_to_xml(n) for n in default_nodes).strip()

        # compiler: keep top-level compiler block when present
        compiler_node = root.find("compiler")
        compiler_xml = _to_xml(compiler_node).strip() if compiler_node is not None else ""

        # assets: merge children from top-level asset blocks
        asset_xml_parts = []
        for asset_node in root.findall("asset"):
            asset_xml_parts.extend(_to_xml(child) for child in list(asset_node))
        asset_xml = "\n".join(asset_xml_parts).strip()

        # joint names: for generating actuators when mjcf has none
        joint_names = list(dict.fromkeys(re.findall(r'<joint\b[^>]*\bname="([^"]+)"', worldbody_inner)))

        # actuators from mjcf (may be empty motor class)
        actuator_node = root.find("actuator")
        actuator_xml = ""
        if actuator_node is not None:
            act_children = list(actuator_node)
            # only use if it has real joint references (not empty motor)
            act_str = "".join(_to_xml(c) for c in act_children)
            if 'joint="' in act_str or "joint='" in act_str:
                actuator_xml = "\n".join(_to_xml(c) for c in act_children).strip()

        return {
            "worldbody": worldbody_inner,
            "default": default_xml,
            "asset": asset_xml,
            "compiler": compiler_xml,
            "actuators": actuator_xml,
            "joint_names": joint_names,
        }
    finally:
        for p in (xacro_preprocessed, urdf_file, mjcf_file):
            if os.path.exists(p):
                try:
                    os.remove(p)
                except OSError:
                    pass


def get_robot_xml():
    """Return robot MJCF parts extracted from .xacro conversion output."""
    return load_robot_from_xacro(ROBOT_XACRO_PATH, ROBOT_PACKAGE_DIRS)


def euler_to_mujoco_quat(roll, pitch, yaw, use_degrees=True):
    """Convert Euler angles (Roll, Pitch, Yaw) to MuJoCo quaternion [w, x, y, z]."""
    r = R.from_euler('xyz', [roll, pitch, yaw], degrees=use_degrees)
    qx, qy, qz, qw = r.as_quat()  # scipy returns [x, y, z, w]
    return np.array([qw, qx, qy, qz])  # MuJoCo uses [w, x, y, z]


def solve_ik_dls_position_only(model, data, desired_pos, ee_site_id, damping=0.01, _debug_step=-1):
    """Position-only DLS IK solver (3DoF). Returns qdot for actuated joints."""
    if model.nu <= 0:
        return None

    # 1. Position error only (3x1)
    current_pos = data.site_xpos[ee_site_id]
    err_pos = desired_pos - current_pos

    # 2. Compute position Jacobian only
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jacSite(model, data, jacp, jacr, ee_site_id)

    # Resolve actuated joint ids (MuJoCo 3 actuator_trnid may be -1; fallback to actuator name)
    jnt_ids = []
    for i in range(model.nu):
        jid = int(model.actuator_trnid[i, 1]) if hasattr(model.actuator_trnid[i, 1], '__int__') else model.actuator_trnid[i, 1]
        if jid < 0:
            aname = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
            jname = aname.replace('_act', '').replace('_ctrl', '') if aname else None
            jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jname) if jname else -1
        jnt_ids.append(jid)
    if any(j < 0 for j in jnt_ids):
        return None

    # Get DOF indices for actuated joints
    act_dofs = [model.jnt_dofadr[jid] for jid in jnt_ids]
    jac = jacp[:, act_dofs]  # Shape: (3, model.nu) - position only

    if _debug_step >= 0 and _debug_step < 3:
        jac_norm = np.linalg.norm(jac)
        print(f"// solve_ik_position: jnt_ids={jnt_ids}, act_dofs={act_dofs}, jac_norm={jac_norm:.6f}")

    # 3. Position-only DLS calculation (3x3 identity matrix)
    J_T = jac.T
    lambda_sq = damping ** 2
    J_J_T_damped = jac @ J_T + lambda_sq * np.eye(3)
    qdot = J_T @ np.linalg.solve(J_J_T_damped, err_pos)

    if _debug_step >= 0 and _debug_step < 2:
        print(f"// solve_ik_position: err_pos={np.round(err_pos, 4)}")

    return qdot, act_dofs


### To-Do: Path Planning ###
# ROBOT_BASE_POS: matches <body name="robot_base" pos="0.75 0.5 0.0"/> in template
ROBOT_BASE_POS = np.array([0.75, 0.25, 0.0])
# WAYPOINTS_IN_BASE_FRAME: True = waypoints relative to robot base, False = world frame
WAYPOINTS_IN_BASE_FRAME = False
# Define waypoint list (XYZ) - frame depends on WAYPOINTS_IN_BASE_FRAME:
# 1) move from start to power_cell,
# 2) make a way point to not collide with red_wall,
# 3) place on target_box
# Tune these points for collision-free motion.
# Do not worry about the grasp & release. If the target is in the range of 0.02 it will automatically grasp. (using the function weld)
# [X, Y, Z, Roll, Pitch, Yaw] - Pitch=180 means gripper pointing down
ROBOT_WAYPOINTS = [
    # 0) Near start pose
    [0.75, 0.25, 0.5],

    [0.75, -0.1, 0.25],

    # # 1) Approach power_cell ans grasp(auto)
    [0.4, -0.1, 0.25],

    # # 2) Move below wall to prevent collision
    [0.75, -0.2, 0.25],

    [0.75, -0.2, 0.45],

    # # 3) Target box top (place power cell on box)
    [1.11, 0.1, 0.42],  
]
WAYPOINT_REACH_THRESHOLD = 0.1



# Magnet (weld) controller state
_ctrl_state = {"weld_active": False, "waypoints": []}


def robot_controller(model, data, step_num):
    """Waypoint-based 6DOF controller with IK and automatic weld logic."""
    global _ctrl_state

    GRASP_THRESHOLD = 0.2
    RELEASE_THRESHOLD = 0.2
    damping = 0.2  # Higher damping for numerical stability

    ee_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "ee_site")
    power_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "power_cell_site")
    target_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "target_center")
    weld_eq_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_EQUALITY, "magnet")

    # Robust fallback for models with missing sites/equality/actuators.
    if ee_site_id < 0 or power_site_id < 0 or target_site_id < 0:
        return

    ee_pos = data.site_xpos[ee_site_id].copy()
    power_pos = data.site_xpos[power_site_id].copy()
    target_pos = data.site_xpos[target_site_id].copy()
    power_body_pos = data.xpos[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "power_cell")].copy()

    # Initialize waypoint queue at the first step.
    if step_num == 0 and not _ctrl_state["waypoints"]:
        _ctrl_state["waypoints"] = [np.array(w, dtype=float) for w in ROBOT_WAYPOINTS]

    # Parse waypoint: [x, y, z, roll, pitch, yaw]
    if _ctrl_state["waypoints"]:
        wp = _ctrl_state["waypoints"][0]
        desired_pos = np.array(wp[:3], dtype=float)
        if WAYPOINTS_IN_BASE_FRAME:
            desired_pos = ROBOT_BASE_POS + desired_pos
        # Check if waypoint reached - move to next waypoint only if there are more
        if np.linalg.norm(ee_pos - desired_pos) < WAYPOINT_REACH_THRESHOLD:
            if len(_ctrl_state["waypoints"]) > 1:
                # Only pop if there's another waypoint to go to
                _ctrl_state["waypoints"].pop(0)
                wp = _ctrl_state["waypoints"][0]
                desired_pos = np.array(wp[:3], dtype=float)
                if WAYPOINTS_IN_BASE_FRAME:
                    desired_pos = ROBOT_BASE_POS + desired_pos
            # else: keep targeting the last waypoint (don't pop it)
    else:
        desired_pos = ee_pos

    # Automatic weld (magnet) logic remains active independent of waypoints.
    dist_to_bolt = np.linalg.norm(ee_pos - power_pos)
    dist_to_target = np.linalg.norm(power_body_pos - target_pos)

    if dist_to_bolt < GRASP_THRESHOLD:
        _ctrl_state["weld_active"] = True
    if dist_to_target < RELEASE_THRESHOLD:
        _ctrl_state["weld_active"] = False

    if weld_eq_id >= 0:
        data.eq_active[weld_eq_id] = 1 if _ctrl_state["weld_active"] else 0

    # DEBUG: ee_pos, desired_pos, x_error (first 5 steps)
    if step_num < 5:
        x_err = desired_pos - ee_pos
        print(f"// ee_pos={ee_pos.round(4)}, desired_pos={desired_pos.round(4)}, x_error={x_err.round(4)}, norm={np.linalg.norm(x_err):.4f}")

    result = solve_ik_dls_position_only(model, data, desired_pos, ee_site_id, damping=damping, _debug_step=step_num)
    if result is None:
        print("// DEBUG: solve_ik_position returned None")
        return
    qdot, act_dofs = result
    ik_gain = 1.0  # High gain for fast convergence
    max_qdot = 2.0  # High max velocity for responsive motion
    qdot = np.clip(qdot, -max_qdot, max_qdot)

    # Always use ACTUAL qpos as base (prevents drift between target and reality)
    q_indices = [model.jnt_qposadr[model.dof_jntid[v]] for v in act_dofs]
    q_current = np.array([data.qpos[q_indices[i]] for i in range(model.nu)], dtype=float)
    q_target = q_current + qdot * ik_gain

    # Clip to joint limits
    for i in range(model.nu):
        jid = model.dof_jntid[act_dofs[i]]
        lo, hi = model.jnt_range[jid, 0], model.jnt_range[jid, 1]
        if hi > lo:
            q_target[i] = np.clip(q_target[i], lo, hi)

    data.ctrl[:] = q_target
    _ctrl_state["q_target"] = q_target

    # DEBUG 2: ctrl vs qpos (first 5 steps)
    if step_num < 5:
        qpos_act = []
        for i in range(model.nu):
            jid = int(model.actuator_trnid[i, 1])
            if jid >= 0:
                qpos_act.append(data.qpos[model.jnt_qposadr[jid]])
        qpos_str = f", qpos(act)={np.round(qpos_act, 4)}" if len(qpos_act) == model.nu else ""
        print(f"// DEBUG step {step_num}: ctrl={np.round(data.ctrl, 4)}{qpos_str}")


# 4. Build & Run Simulation

`build_simulation_model()` assembles the full XML from the template and robot parts, sanitizes URDF conversion artifacts (mesh scale, freejoint), injects `ee_site` when missing, resolves texture paths, and loads the MuJoCo model. `run_simulation()` steps the sim, renders frames from two cameras, and displays the video via mediapy.

In [ ]:
# 4) Build / Run Functions

def build_simulation_model():
    robot_parts = get_robot_xml()
    if isinstance(robot_parts, dict):
        robot_xml = robot_parts.get("worldbody", "")
        robot_default_xml = robot_parts.get("default", "")
        robot_asset_xml = robot_parts.get("asset", "")
        robot_compiler_xml = robot_parts.get("compiler", "")
        robot_actuators = robot_parts.get("actuators", "")
        robot_joint_names = robot_parts.get("joint_names", [])
    else:
        robot_xml = robot_parts
        robot_default_xml = ""
        robot_asset_xml = ""
        robot_compiler_xml = ""
        robot_actuators = ""
        robot_joint_names = []

    # Move mesh scale from <geom ... scale="..."> to corresponding <asset><mesh ... scale="..."/>
    # because MuJoCo rejects scale on geom for mesh geoms.
    geom_mesh_scales = {}
    for m in re.finditer(r'<geom\b[^>]*\bmesh="([^"]+)"[^>]*\bscale="([^"]+)"[^>]*/>', robot_xml):
        mesh_name, scale_val = m.group(1), m.group(2)
        geom_mesh_scales.setdefault(mesh_name, scale_val)

    if geom_mesh_scales and robot_asset_xml:
        for mesh_name, scale_val in geom_mesh_scales.items():
            # add scale to mesh asset if missing
            pat_no_scale = rf'(<mesh\b[^>]*\bname="{re.escape(mesh_name)}"(?![^>]*\bscale=)[^>]*)/>'
            robot_asset_xml = re.sub(pat_no_scale, rf'\1 scale="{scale_val}"/>', robot_asset_xml)

    # remove invalid geom scale attributes after transferring scale to assets
    robot_xml = re.sub(r'(<geom\b[^>]*?)\s+scale="[^"]*"', r'\1', robot_xml)

    # Robot is injected inside robot_base, so it must not have free joint DOFs
    # (free joints are allowed only on worldbody-level top bodies in MuJoCo).
    robot_xml = re.sub(r'\s*<freejoint\b[^>]*/>\s*', ' ', robot_xml)
    robot_xml = re.sub(r'\s*<joint\b[^>]*\btype="free"[^>]*/>\s*', ' ', robot_xml)
    robot_xml = re.sub(r"\s*<joint\b[^>]*\btype='free'[^>]*/>\s*", ' ', robot_xml)

    # Ensure controller-required site exists for xacro-loaded robots.
    if 'name="ee_site"' not in robot_xml:
        ee_site_xml = '\n            <site name="ee_site" pos="0 0 0" size="0.01" rgba="1 0 0 1"/>'
        inserted = False
        for body_name in ("limb6_1", "end_effector", "tool0", "ee_link"):
            pattern = rf'(<body\b[^>]*name="{body_name}"[^>]*>)'
            robot_xml, n = re.subn(pattern, rf'\1{ee_site_xml}', robot_xml, count=1)
            if n > 0:
                inserted = True
                break
        if not inserted:
            robot_xml = ee_site_xml + "\n" + robot_xml

    full_xml = ENV_XML_TEMPLATE.replace("<!-- ROBOT_COMPILER_PLACEHOLDER -->", robot_compiler_xml)
    full_xml = full_xml.replace("<!-- ROBOT_DEFAULT_PLACEHOLDER -->", robot_default_xml)
    full_xml = full_xml.replace("<!-- ROBOT_ASSET_PLACEHOLDER -->", robot_asset_xml)
    full_xml = full_xml.replace("<!-- ROBOT_PLACEHOLDER -->", robot_xml)

    # Inject actuators for the loaded robot
    valid_joints = set(robot_joint_names)
    filtered_robot_actuators = ""
    if robot_actuators:
        lines = [ln for ln in robot_actuators.splitlines() if ln.strip()]
        if valid_joints:
            # Keep only actuator entries targeting joints present in injected robot XML.
            kept = []
            for ln in lines:
                m = re.search(r'\bjoint="([^"]+)"', ln)
                if m is None or m.group(1) in valid_joints:
                    kept.append(ln)
            filtered_robot_actuators = "\n".join(kept).strip()
        else:
            # If joint discovery is empty, trust converted actuator block as-is.
            filtered_robot_actuators = "\n".join(lines).strip()

    # Use position actuators from joint names; urdf2mjcf motor class causes all to bind to joint6.
    # High kp for strong position control, high forcerange to overcome gravity
    if robot_joint_names:
        actuator_block = "<actuator>\n" + "\n".join(
            f'        <position name="{jn}_act" joint="{jn}" kp="2000" kv="200" forcerange="-2000 2000"/>'
            for jn in robot_joint_names
        ) + "\n    </actuator>"
    elif filtered_robot_actuators:
        actuator_block = "<actuator>\n" + filtered_robot_actuators + "\n    </actuator>"
    else:
        actuator_block = ""
    full_xml = full_xml.replace("<!-- ROBOT_ACTUATOR_PLACEHOLDER -->", actuator_block)

    # If ee_site is missing (e.g., robot injection disabled), remove magnet equality.
    if 'name="ee_site"' not in full_xml:
        full_xml = re.sub(r"<equality>.*?</equality>", "", full_xml, flags=re.DOTALL)

    # Load star texture (prefer PNG for MuJoCo compatibility)
    try:
        from PIL import Image
        jpg_path = os.path.abspath("8k_stars_milky_way.jpg")
        png_path = os.path.abspath("8k_stars_milky_way.png")
        if os.path.exists(jpg_path) and (not os.path.exists(png_path) or os.path.getmtime(jpg_path) > os.path.getmtime(png_path)):
            Image.open(jpg_path).convert("RGB").save(png_path)
        tex_path = png_path if os.path.exists(png_path) else jpg_path
    except Exception:
        tex_path = os.path.abspath("8k_stars_milky_way.jpg")
    full_xml = full_xml.replace("8k_stars_milky_way.png", tex_path)

    inside_sphere_obj = os.path.abspath("inside_sphere.obj")
    ensure_inside_sphere_obj(inside_sphere_obj)
    full_xml = full_xml.replace("INSIDE_SPHERE_OBJ_PATH", inside_sphere_obj)

    try:
        from PIL import Image
        concrete_jpg = os.path.abspath("concrete.jpg")
        concrete_png = os.path.abspath("concrete.png")
        if os.path.exists(concrete_jpg) and (not os.path.exists(concrete_png) or os.path.getmtime(concrete_jpg) > os.path.getmtime(concrete_png)):
            Image.open(concrete_jpg).convert("RGB").save(concrete_png)
        concrete_path = concrete_png if os.path.exists(concrete_png) else concrete_jpg
    except Exception:
        concrete_path = os.path.abspath("concrete.jpg")
    full_xml = full_xml.replace("CONCRETE_TEX_PATH", concrete_path)

    DEBUG_XML = False
    if DEBUG_XML:
        print("// debug joints:", len(robot_joint_names))
        print("// debug actuator_block:", "yes" if actuator_block else "no")
        print("// debug has_freejoint:", "<freejoint" in full_xml)

    try:
        model = mujoco.MjModel.from_xml_string(full_xml)
    except ValueError as e:
        # Fallback: if template actuators don't match converted robot joint names, drop them and retry.
        if "joint" in str(e).lower() and "actuator" in full_xml:
            full_xml_retry = re.sub(r"<actuator>.*?</actuator>", "", full_xml, flags=re.DOTALL)
            model = mujoco.MjModel.from_xml_string(full_xml_retry)
        else:
            raise

    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, height=720, width=960)
    power_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "power_cell")
    return model, data, renderer, power_body_id


def run_simulation(duration=5.0, fps=60, debug_steps=200):
    if not HAVE_MUJOCO:
        print("// SKIPPING SIMULATION: install mujoco/mediapy to run.")
        return

    print("// INITIALIZING DEATH STAR SIMULATION CORE...")
    _ctrl_state["weld_active"] = False
    _ctrl_state["waypoints"] = [np.array(w, dtype=float) for w in ROBOT_WAYPOINTS]
    _ctrl_state["q_target"] = None
    model, data, renderer, power_body_id = build_simulation_model()

    total_steps = int(duration / model.opt.timestep)
    render_step = max(1, int(1.0 / (fps * model.opt.timestep)))
    frames = []

    print(f"// SYSTEM READY. Simulating {duration} seconds...")
    mujoco.mj_forward(model, data)

    # Initialize joints to a reasonable starting configuration (slightly bent)
    if model.nq == 13 and model.nu == 6:
        # Give joints initial angles to form a stable starting pose
        data.qpos[7:13] = [0.0, 0.3, -0.5, 0.3, -0.3, 0.0]  # bent arm configuration
        mujoco.mj_forward(model, data)

    # DEBUG 1: actuator / DOF counts
    print(f"// DEBUG: model.nu (actuators)={model.nu}, model.nv (vel DOF)={model.nv}, model.nq={model.nq}")

    # geom ID for red_wall color (yellow when touching, red when not)
    red_wall_geom_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, "red_wall_geom")

    for i in range(total_steps):
        robot_controller(model, data, i)
        mujoco.mj_step(model, data)

        # red_wall: yellow when touching another object, red when not
        if red_wall_geom_id >= 0:
            touching = False
            for ci in range(data.ncon):
                con = data.contact[ci]
                if con.geom1 == red_wall_geom_id or con.geom2 == red_wall_geom_id:
                    touching = True
                    break
            if touching:
                model.geom_rgba[red_wall_geom_id] = [0.9, 0.9, 0.1, 1]  # yellow
            else:
                model.geom_rgba[red_wall_geom_id] = [0.9, 0.1, 0.1, 1]  # red

        if i < debug_steps:
            pz = data.xpos[power_body_id][2]
            if data.ncon > 0:
                names = []
                for ci in range(data.ncon):
                    con = data.contact[ci]
                    g1 = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, con.geom1)
                    g2 = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, con.geom2)
                    g1 = g1 if g1 is not None else f"geom#{con.geom1}"
                    g2 = g2 if g2 is not None else f"geom#{con.geom2}"
                    names.append(f"{g1} <-> {g2}")
                print(f"// step {i} z={pz:.3f} contacts: " + ", ".join(names))
            else:
                print(f"// step {i} z={pz:.3f} contacts: none")

        if i % render_step == 0:
            renderer.update_scene(data, camera="diag_cam")
            img_main = renderer.render()
            renderer.update_scene(data, camera="power_cam")
            img_power = renderer.render()
            frames.append(np.vstack([img_main, img_power]))

    print("// SIMULATION COMPLETE. COMPILING HOLORECORDING...")
    print(f"// Total frames captured: {len(frames)}")
    
    if len(frames) == 0:
        print("// ERROR: No frames were captured!")
        return
    
    # Save video to file first (more reliable in Colab)
    video_path = "simulation_output.mp4"
    media.write_video(video_path, frames, fps=fps)
    print(f"// Video saved to: {video_path}")
    
    # Display video in Colab using IPython
    from IPython.display import HTML
    from base64 import b64encode
    
    with open(video_path, "rb") as f:
        video_data = f.read()
    video_b64 = b64encode(video_data).decode()
    
    display(HTML(f'''
    <video width="960" controls autoplay loop>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''))

# 5. Execute

Run the simulation. Ensure cells 1–4 have been executed in order.

**Google Colab 사용시 필수 확인사항:**
1. Google Drive가 마운트되었는지 확인
2. `GDRIVE_PROJECT_PATH`가 올바른 경로로 설정되었는지 확인
3. 필요한 에셋 파일들 (`8k_stars_milky_way.jpg`, `concrete.jpg`)이 프로젝트 폴더에 있는지 확인
4. 로봇 URDF 파일들이 지정된 경로에 존재하는지 확인

In [ ]:
# 5) Execute
run_simulation(duration=5.0, fps=60, debug_steps=200)